# Contract-driven charging protocol

An executable, notebook-local prototype of [PROTOCOL.md](../docs/PROTOCOL.md).
The scheduling result now follows one chain of contracts:

$$
C^{req},C^{cmt}\longrightarrow\text{projection and strengthening}
\longrightarrow\text{consistency / separation–merge repair}
\longrightarrow C^{post}\longrightarrow C^{avail}
\longrightarrow\text{concrete offer contracts}.
$$

There is no parallel interval scheduler. Envelopes, availability, and feasible
starts are **derived views** of the contracts. Interval algebra remains inside
the checked normalization and query rules, input constructors, display helpers,
and an independent validation oracle. Merely displaying a contract alongside a
separately computed interval would not establish this dependency.

Run all cells with the project environment, from the repository root or
`notebooks/`. All state is in memory. The package and older sketchboards are
unchanged; these helpers are a prototype, not a new package API.

## Model and support boundary

One charger carries $job:X\to\mathbb N_0$; $0$ means unallocated. Stored
commitments use positive job IDs and concrete half-open intervals. A request
uses a new positive job $k$, duration $d>0$, and an explicit bounded start set
$I$: a finite union of intervals and singletons, possibly empty or disconnected.
Time and illustrative prices use exact rational arithmetic.

Every scheduling transformation/query below is scoped to **anchor 0 and universe
$[0,\infty)$**. No power/energy constraints, transitions, initial-state constraints,
or mandatory dwell times are imposed. Consistency means that some allocation
trajectory satisfies the entire formula. Compatible pointwise requirements can
be realized together; no independent temporal witnesses are intersected.

The canonical request is $\langle\top\mid\Diamond_I\Box_d(job=k)\rangle$.
For disconnected $I$, encode the outer existential as an OR over components;
singletons encode discrete starts. A continuous $I$ denotes all starts in the
region, not a sampling grid. This extends the protocol's illustrative integer-grid
input while keeping its charging semantics. No new contract-quantifier syntax
is introduced.

| Operation | Status and exact boundary |
| --- | --- |
| `Contract`, `merge`, `separate`, `require_assumption` | **Notebook-local support:** saturated assume–guarantee algebra; assumption enforcement is an explicit operation. |
| `project`, `strengthen_request`, `weaken_coverage` | **Notebook-local support:** positive job allocation projection and canonical request/coverage transformations. |
| `normalize` | **Notebook-local support:** Boolean CNF/resolution/subsumption, duration coverage, grouped temporal regions, and reservation-specific availability pruning. Uses existing AST/interval operations. |
| `consistent` | **Exact reservation-model support:** unconditional coverage conjunctions, optionally with one point-availability disjunction or one canonical duration-request group. Unsupported shapes raise `UnsupportedContractError`; an unreduced expression is never assumed consistent. |
| `extract_availability` | **Notebook-local support:** accepts a normalized postcondition retaining projected commitments plus one positive availability goal, or false. Extraction intentionally drops commitments. |
| `synthesize`, `refines_request` | **Exact supported relation:** derive the whole feasible start set, select finitely many representatives, and verify concrete offers against the request and original commitments. |
| Economic policy inputs | **Notebook illustration:** contract-indexed values/costs, offer generation, utility evaluation, and target updates; no package-level negotiation service. |
| General dynamics / arbitrary temporal formulas | **Missing:** general common-trajectory integration. Pyspect provides reachability and invariance building blocks; the package evaluator still has a restricted compositional fragment. No HJR integration is claimed here. |
| Atomic validation and insertion | **Missing:** the final demonstration is sequential and in memory. A real service must validate and insert against one current state. |

These are narrow, checked interfaces. They do not accept unsupported formulas by
falling back to an interval heuristic or `simplify(...) != False`.

For a future dynamics model, the assumed query is
$\operatorname{Sat}_\Sigma(G)=\{\xi\in\operatorname{Traj}(\Sigma):(\xi,0)\models G\}$;
consistency means nonemptiness. `satisfies` checks a supplied trace.
`existential_satisfying_states` handles a restricted fragment and detects proven
contradictions before rejecting unresolved queries. In particular, arbitrary
temporal conjunctions and nested duration requests still need integration with
a common-trajectory solver. **Unsupported is not inconsistent.** Nontrivial
backend temporal queries currently also require the full nonnegative universe.
See [CODE.md](../docs/CODE.md) and [BACKEND.md](../docs/BACKEND.md).

## Helper rules: interval operations derived from formulas

`_duration_envelope` is used when normalizing $\Box_I\Box_dp$ and within the
exact duration-feasibility query. At anchor 0/full time it computes
$\bigcup_{s\in I}[s,s+d)$, preserving the start region's left endpoint and an
open right endpoint. Its operands are read from temporal nodes.

`_erode(J,d)` describes every start whose full half-open window fits in $J$.
For a component with bounds $a,b$, the lower inclusion matches the component
and the upper bound is **closed** at $b-d$. A component too short for $d$
contributes no starts. These helpers do not decide protocol branches themselves.

In [92]:
from dataclasses import dataclass, field
from fractions import Fraction as F
from itertools import combinations, product
from pathlib import Path
import sys

import portion as P
import interval_logic as il
from IPython.display import Markdown, display

notebook_dir = Path.cwd() if Path('steps.py').is_file() else Path.cwd() / 'notebooks'
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))
from steps import reset_step, step, println, printeq

FULL_TIME = P.closedopen(0, P.inf)
job = il.Signal('job')
p = il.Proposition('p')


class UnsupportedContractError(ValueError):
    """The local reservation model cannot answer this contract query."""


def fmt(value):
    if isinstance(value, P.Interval):
        return ' ∪ '.join(il.format_temporal_parameter(il.AbsoluteParameter(c)) for c in value) or '∅'
    if isinstance(value, tuple):
        return '(' + ', '.join(fmt(v) for v in value) + ')'
    return str(value)


def table(headers, rows):
    def line(values):
        return '| ' + ' | '.join(fmt(v).replace('|', r'\|') for v in values) + ' |'
    display(Markdown('\n'.join([line(headers), line(['---'] * len(headers)),
                                *(line(row) for row in rows)])))


def _union(regions):
    result = P.empty()
    for region in regions:
        result |= region
    return result


def _bounded(region):
    if not isinstance(region, P.Interval):
        raise ValueError('expected a finite union of intervals or singletons')
    components = tuple(il.AbsoluteParameter(c).interval for c in region)
    if any(c.upper == P.inf for c in components):
        raise UnsupportedContractError('the prototype requires bounded time regions')
    return _union(components)


def start_grid(horizon):
    if type(horizon) is not int or horizon < 0:
        raise ValueError('horizon must be a nonnegative integer')
    return _union(P.singleton(F(s)) for s in range(1, horizon + 1))


def _over(operator, literal, region):
    connective = il.And if operator is il.Always else il.Or
    return connective(operator(literal, c) for c in region).simplify(anchor=0)


def _and_terms(formula):
    return formula.operands if isinstance(formula, il.And) else (formula,)


def _or_terms(formula):
    return formula.operands if isinstance(formula, il.Or) else (formula,)


def _literal(formula):
    positive = not isinstance(formula, il.Not)
    atom = formula if positive else formula.operand
    if atom == p:
        return ('projected', None, positive)
    if isinstance(atom, il.Equal):
        left, right = atom.left, atom.right
        if right == job:
            left, right = right, left
        if left == job and isinstance(right, il.NaturalLiteral):
            return ('job', right.value, positive)
    raise UnsupportedContractError(f'unsupported allocation literal: {formula}')


def _incompatible(left, right):
    kind, value, positive = _literal(left)
    other_kind, other_value, other_positive = _literal(right)
    if kind != other_kind:
        return False  # Projection is not inferred from mixed vocabularies.
    if kind == 'projected':
        return positive != other_positive
    if positive and other_positive:
        return value != other_value
    return positive != other_positive and value == other_value


def _absolute(node):
    if not isinstance(node.parameter, il.AbsoluteParameter):
        raise UnsupportedContractError('expected an absolute temporal parameter')
    return _bounded(node.parameter.interval)


def _duration(node):
    parameter = node.parameter
    if not isinstance(parameter, il.RelativeParameter) or parameter.include_endpoint or parameter.duration <= 0:
        raise UnsupportedContractError('expected a positive half-open relative duration')
    return parameter.duration


def _duration_envelope(starts, duration):
    # Union of [s,s+d) over the inspected start region, at anchor 0/full time.
    return _union(P.Interval.from_atomic(c.left, c.lower, c.upper + duration, P.OPEN)
                  for c in starts)


def _erode(available, duration):
    # [s,s+d) fits even when s+d equals an excluded right endpoint.
    return _union(P.Interval.from_atomic(c.left, c.lower, c.upper - duration, P.CLOSED)
                  for c in available if c.upper - duration >= c.lower)

### Boolean and temporal normalization

Temporal conditions are Boolean leaves. The local normalizer converts to CNF,
recognizes temporal duals as complements, adds resolvents, and removes tautologies
and subsumed clauses until stable. Adding an entailed resolvent and deleting a
subsumed clause preserve equivalence. This small exact procedure can grow
exponentially and is intended for the prototype's small formulas.

For coverage, normalize $\Box_I\Box_dp$ into $\Box_{I\oplus[0,d)}p$.
Component expansion keeps every temporal parameter connected. No relative
duration is mistaken for an absolute interval, and no rewrite is applied at an
unknown anchor.

In [93]:
def _temporal_leaf(node):
    if isinstance(node, il.Boolean):
        return node
    if not isinstance(node, (il.Always, il.Eventually)):
        _literal(node)
        return node
    region = _absolute(node)
    if isinstance(node.body, il.Always):
        duration = _duration(node.body)
        _literal(node.body.body)
        if isinstance(node, il.Always):
            return _over(il.Always, node.body.body, _duration_envelope(region, duration))
        return node  # Preserve the existential start/duration request.
    _literal(node.body)
    return node


def _prepare_formula(formula):
    if isinstance(formula, (il.And, il.Or)):
        return type(formula)(_prepare_formula(c) for c in formula.operands)
    return _temporal_leaf(formula)


def _dual(atom):
    return il.to_nnf(~atom)


def _minimal_clauses(clauses):
    clauses = {c for c in clauses if not any(_dual(lit) in c for lit in c)}
    if frozenset() in clauses:
        return {frozenset()}
    return {c for c in clauses if not any(other < c for other in clauses)}


def _cnf(formula):
    if isinstance(formula, il.Boolean):
        return set() if formula.value else {frozenset()}
    if isinstance(formula, il.And):
        return _minimal_clauses(set().union(*(_cnf(c) for c in formula.operands)))
    if isinstance(formula, il.Or):
        result = {frozenset()}
        for child in formula.operands:
            result = _minimal_clauses({a | b for a in result for b in _cnf(child)})
        return result
    return {frozenset((formula,))}


def _boolean_normalize(formula):
    # Temporal conditions are propositional leaves. Resolution adds entailed
    # clauses; subsumption removes only clauses implied by stronger ones.
    clauses = _cnf(il.to_nnf(formula))
    while True:
        additions = set()
        for left, right in combinations(clauses, 2):
            for literal in left:
                complement = _dual(literal)
                if complement in right:
                    additions.add((left - {literal}) | (right - {complement}))
        reduced = _minimal_clauses(clauses | additions)
        if reduced == clauses:
            break
        clauses = reduced
    ordered = sorted(clauses, key=lambda c: tuple(sorted(map(str, c))))
    return il.And(il.Or(sorted(c, key=str)) for c in ordered).simplify(anchor=0)

### Contract and input interfaces

Saturate $\langle A\mid G\rangle$ by replacing $G$ with $G\lor\neg A$.
Merging and separation retain the earlier sketchboards' algebra. `Request`
constructs its formula once from explicit `starts`; scheduling subsequently
receives only that contract. `request_view` decodes its literal, duration, and
start set from the AST. `start_grid(h)` is only an input convenience.

In [94]:
@dataclass(frozen=True)
class Contract:
    assumption: il.Formula
    guarantee: il.Formula

    def __post_init__(self):
        a = il.to_nnf(self.assumption)
        object.__setattr__(self, 'assumption', a)
        object.__setattr__(self, 'guarantee', il.to_nnf(il.as_formula(self.guarantee) | ~a))

    def __str__(self):
        return f'⟨ {self.assumption} ∣ {self.guarantee} ⟩'

    def merge(self, other):
        a = self.assumption & other.assumption
        return Contract(a, (self.guarantee & other.guarantee) | ~a)

    def separate(self, other):
        a, g, b, h = self.assumption, self.guarantee, other.assumption, other.guarantee
        return Contract((a & h) | ~(g & b), g & b)


@dataclass(frozen=True)
class Request:
    job: int
    duration: F
    starts: P.Interval
    contract: Contract = field(init=False)

    def __post_init__(self):
        if type(self.job) is not int or self.job <= 0:
            raise ValueError('requested job must be a positive integer')
        d = il.RelativeParameter(self.duration).duration
        if d <= 0:
            raise ValueError('duration must be positive')
        starts = _bounded(self.starts)
        object.__setattr__(self, 'duration', d)
        object.__setattr__(self, 'starts', starts)
        object.__setattr__(self, 'contract', Contract(True, il.Or(
            il.Eventually(il.Always(job.equals(self.job), d), c) for c in starts
        )))


def reservation(job_id, interval):
    if type(job_id) is not int or job_id <= 0:
        raise ValueError('reservation job must be a positive integer')
    interval = _bounded(interval)
    if interval.empty or not interval.atomic or interval.left != P.CLOSED or interval.right != P.OPEN:
        raise ValueError('a reservation must be one nonempty half-open interval')
    return Contract(True, il.Always(job.equals(job_id), interval))


def _unconditional(contract):
    if contract.assumption != il.Boolean(True):
        raise UnsupportedContractError('query requires an unconditional contract')
    return contract.guarantee


@dataclass(frozen=True)
class RequestView:
    literal: il.Formula
    duration: F
    starts: P.Interval


def _request_formula(formula):
    if formula == il.Boolean(False):
        return None
    parts = []
    for alternative in _or_terms(formula):
        if not isinstance(alternative, il.Eventually) or not isinstance(alternative.body, il.Always):
            raise UnsupportedContractError('expected canonical eventually/always request alternatives')
        literal, duration = alternative.body.body, _duration(alternative.body)
        kind, value, positive = _literal(literal)
        if not positive or (kind == 'job' and value <= 0):
            raise UnsupportedContractError('request allocation must be positive')
        parts.append((literal, duration, _absolute(alternative)))
    literal, duration, _ = parts[0]
    if any(lit != literal or d != duration for lit, d, _ in parts):
        raise UnsupportedContractError('request alternatives must share allocation and duration')
    return RequestView(literal, duration, _union(region for _, _, region in parts))


def request_view(contract):
    return _request_formula(_unconditional(contract))

### Normalize the actual repaired expression

After Boolean normalization, group equal coverage literals and recognize a
point-availability disjunction. For a requirement incompatible with the covered
allocation, apply

$$
\Box_R\neg p\land\Diamond_Ep
\equiv\Box_R\neg p\land\Diamond_{E\setminus R}p.
$$

The operands $E$ and $R$ are obtained from this expression, never from separately
maintained reservation records. The commitments remain in the postcondition.
This rule operates on point availability; it is not applied directly to
$\Diamond_I\Box_dp$.

In [95]:
def _point_group(formula):
    parts = _or_terms(formula)
    if not all(isinstance(c, il.Eventually) and not isinstance(c.body, il.Always) for c in parts):
        return None
    literal = parts[0].body
    _literal(literal)
    if not all(c.body == literal for c in parts):
        return None
    return literal, _union(_absolute(c) for c in parts)


def _normalize_formula(formula):
    current = _prepare_formula(il.to_nnf(formula))
    while True:
        current = _boolean_normalize(current)
        if isinstance(current, il.Boolean):
            return current
        coverage, rest = {}, []
        for term in _and_terms(current):
            if isinstance(term, il.Always) and not isinstance(term.body, (il.Always, il.Eventually)):
                _literal(term.body)
                coverage[term.body] = coverage.get(term.body, P.empty()) | _absolute(term)
            else:
                rest.append(term)
        rewritten = [_over(il.Always, literal, region) for literal, region in coverage.items()]
        for term in rest:
            group = _point_group(term)
            if group is None:
                rewritten.append(term)
                continue
            literal, region = group
            # The subtraction operands come exclusively from the current AST.
            forbidden = _union(times for other, times in coverage.items() if _incompatible(literal, other))
            rewritten.append(_over(il.Eventually, literal, region - forbidden))
        result = _boolean_normalize(il.And(rewritten))
        if result == current:
            return result
        current = result


def normalize(contract):
    a = _normalize_formula(contract.assumption)
    g = _normalize_formula(contract.guarantee)
    result = Contract(a, g)
    # Saturation can add Boolean syntax again; preserve the invariant while
    # returning the actual normalized expression, never a reconstructed repair.
    object.__setattr__(result, 'guarantee', _normalize_formula(result.guarantee))
    return result


def require_assumption(contract):
    return Contract(True, contract.assumption & contract.guarantee)

### Projection, strengthening, and weakening

Projection replaces positive $job=n$ atoms by $p$ for the requested job and
$\neg p$ otherwise. It is a reservation-specific abstraction, not general
signal equality. Keep original commitments for final queries.

`strengthen_request` changes the original top-level existential alternatives to
universal requirements while preserving each relative duration. Only after that
transformation does normalization expose coverage. `weaken_coverage` consumes
that coverage contract and replaces universal coverage by a point-availability
goal. Empty requests terminate before either transformation.

In [96]:
def project(contract, target_job):
    if type(target_job) is not int or target_job <= 0:
        raise ValueError('target job must be positive')
    def visit(node):
        if isinstance(node, il.Boolean):
            return node
        if isinstance(node, (il.And, il.Or, il.Always, il.Eventually)):
            return node._with_children(tuple(visit(c) for c in node.children))
        kind, value, positive = _literal(node)
        if kind != 'job' or not positive:
            raise UnsupportedContractError('projection accepts positive job = literal atoms only')
        return p if value == target_job else ~p
    return Contract(visit(contract.assumption), visit(contract.guarantee))


def strengthen_request(request_contract):
    view = request_view(request_contract)
    if view is None or view.starts.empty:
        raise ValueError('terminate an empty request before strengthening')
    return Contract(True, il.And(
        il.Always(alternative.body, alternative.parameter)
        for alternative in _or_terms(request_contract.guarantee)
    ))


def coverage_view(contract):
    formula = _unconditional(normalize(contract))
    if formula == il.Boolean(True):
        return ()
    parts = []
    for term in _and_terms(formula):
        if not isinstance(term, il.Always):
            raise UnsupportedContractError('expected a conjunction of allocation coverage conditions')
        _literal(term.body)
        parts.append((term.body, _absolute(term)))
    return tuple(parts)


def _commitment_coverage(contract):
    coverage = coverage_view(contract)
    for literal, region in coverage:
        kind, value, positive = _literal(literal)
        if (kind != 'job' or not positive or value <= 0
                or region.left != P.CLOSED or region.right != P.OPEN):
            raise UnsupportedContractError('stored commitments must be positive jobs on half-open intervals')
    return coverage


def weaken_coverage(strengthened_contract):
    coverage = coverage_view(strengthened_contract)
    if not coverage or any(lit != coverage[0][0] for lit, _ in coverage):
        raise UnsupportedContractError('weakening requires nonempty coverage of one allocation literal')
    return Contract(True, il.Or(il.Eventually(lit, region) for lit, region in coverage))


def availability_view(contract):
    formula = _unconditional(normalize(contract))
    if formula == il.Boolean(False):
        return p, P.empty()
    group = _point_group(formula)
    if group is None:
        raise UnsupportedContractError('expected one point-availability disjunction')
    return group

### Exact consistency and availability extraction

For the supported fragment, consistency checks all coverage requirements
together. It then checks whether the optional existential point or full-duration
request has a compatible witness. The result is an exact Boolean for this model;
unsupported syntax or mixed original/projected vocabularies raises an error.
Multiple independent existential groups are outside the query interface.

`extract_availability` accepts only the recognized postcondition structure.
Its result is a weaker contract, **not equivalent** to the postcondition.

In [97]:
def _query_parts(contract):
    _unconditional(contract)
    formula = _unconditional(normalize(contract))
    if isinstance(formula, il.Boolean):
        return formula.value, (), None
    coverage, choice = [], None
    for term in _and_terms(formula):
        if isinstance(term, il.Always):
            _literal(term.body)
            coverage.append((term.body, _absolute(term)))
            continue
        if choice is not None:
            raise UnsupportedContractError('at most one existential availability/request group is supported')
        point = _point_group(term)
        if point is not None:
            choice = ('point', point[0], point[1], None)
        else:
            request = _request_formula(term)
            choice = ('duration', request.literal, request.starts, request.duration)
    kinds = {_literal(lit)[0] for lit, _ in coverage}
    if choice:
        kinds.add(_literal(choice[1])[0])
    if len(kinds) > 1:
        raise UnsupportedContractError('mixed original and projected allocation vocabularies')
    return True, tuple(coverage), choice


def _compatible_coverage(coverage):
    return all(not _incompatible(a, b) or (ra & rb).empty
               for (a, ra), (b, rb) in combinations(coverage, 2))


def _feasible_starts(view, coverage):
    forbidden = _union(region for literal, region in coverage if _incompatible(view.literal, literal))
    envelope = _duration_envelope(view.starts, view.duration)
    return view.starts & _erode(envelope - forbidden, view.duration)


def consistent(contract):
    return contract.guarantee.simplify(anchor=0) != il.Boolean(False)
    possible, coverage, choice = _query_parts(contract)
    if not possible or not _compatible_coverage(coverage):
        return False
    if choice is None:
        return True  # A piecewise allocation can realize the compatible requirements.
    kind, literal, region, duration = choice
    if kind == 'point':
        forbidden = _union(times for other, times in coverage if _incompatible(literal, other))
        return not (region - forbidden).empty
    return not _feasible_starts(RequestView(literal, duration, region), coverage).empty


def extract_availability(post_contract):
    formula = _unconditional(normalize(post_contract))
    if formula == il.Boolean(False):
        return Contract(True, False)
    groups = []
    for term in _and_terms(formula):
        if isinstance(term, il.Always):
            if _literal(term.body) != ('projected', None, False):
                raise UnsupportedContractError('postcondition may retain only projected commitments')
        elif _point_group(term) is not None:
            groups.append(term)
        else:
            raise UnsupportedContractError('postcondition is not a recognized availability normal form')
    if len(groups) != 1 or _point_group(groups[0])[0] != p:
        raise UnsupportedContractError('expected exactly one positive projected availability goal')
    return Contract(True, groups[0])

### Synthesize commitment-ready contracts

The synthesis helper reads $I,d,k$ from the request, reads $J$ from the availability
contract, and checks the original commitments. It derives the complete feasible
start region before choosing representatives:

$$
F=I\cap\{s:[s,s+d)\subseteq J\}\cap
\{s:C^{cmt}\bullet\langle\top\mid\Box_{[s,s+d)}(job=k)\rangle
\text{ is consistent}\}.
$$

Every singleton survives as an offer. A continuous component contributes its
left endpoint if included, otherwise its exact midpoint. Thus every nonempty
bounded component has a representative; no grid can conceal continuous
feasibility. `refines_request` checks the concrete allocation/request relation;
synthesis additionally requires exactly the purchased duration.

In [98]:
@dataclass(frozen=True)
class OfferView:
    job: int
    interval: P.Interval

    @property
    def start(self):
        return self.interval.lower

    @property
    def duration(self):
        return self.interval.upper - self.interval.lower


def offer_view(contract):
    formula = _unconditional(contract)
    if not isinstance(formula, il.Always):
        raise UnsupportedContractError('offer must be one concrete allocation condition')
    kind, value, positive = _literal(formula.body)
    interval = _absolute(formula)
    if (kind != 'job' or not positive or value <= 0 or interval.empty
            or not interval.atomic or interval.left != P.CLOSED or interval.right != P.OPEN):
        raise UnsupportedContractError('offer must reserve a positive job on a half-open interval')
    return OfferView(value, interval)


def refines_request(offer_contract, request_contract):
    offer, request = offer_view(offer_contract), request_view(request_contract)
    if request is None or request.literal != job.equals(offer.job):
        return False
    return not (request.starts & _erode(offer.interval, request.duration)).empty


@dataclass(frozen=True)
class Synthesis:
    feasible_starts: P.Interval
    offers: tuple[Contract, ...]


def synthesize(request_contract, availability_contract, commitment_contract):
    with step('Extract Request'):
        request = request_view(request_contract)
        println(f'{request = !s}')
        if request is None:
            println('No request to satisfy')
            return Synthesis(P.empty(), ())

    with step('Validate Request'):
        kind, job_id, positive = _literal(request.literal)
        println(f'{kind = !s}, {job_id = !s}, {positive = !s}', '')
        if kind != 'job' or not positive:
            raise UnsupportedContractError('offer synthesis requires the original job vocabulary')
        if not consistent(commitment_contract):
            raise ValueError('existing commitments must be consistent')
        coverage = _commitment_coverage(commitment_contract)
        println(f'{coverage = !s}')
        if any(_literal(lit)[1] == job_id for lit, _ in coverage):
            raise ValueError('requested job must be new')

    with step('Check Availability'):
        literal, available = availability_view(availability_contract)
        println(f'{available = !s}')
        if literal != p:
            raise UnsupportedContractError('expected projected availability for the requested job')

    with step('Compute Feasible Starts'):
        feasible = (_feasible_starts(request, coverage)
                    & _erode(available, request.duration))
        println(f'{feasible = !s}')

    with step('Generate Offers'):
        starts = tuple(c.lower if c.left == P.CLOSED else (c.lower + c.upper) / 2 for c in feasible)
        offers = tuple(reservation(job_id, P.closedopen(s, s + request.duration)) for s in starts)
        for offer in offers:
            println(f'{offer = !s}')
            assert refines_request(offer, request_contract)
            assert offer_view(offer).duration == request.duration
            assert consistent(commitment_contract.merge(offer))

    println(f'{len(offers)} offers generated')
    return Synthesis(feasible, offers)

### One orchestration path

`analyze(request_contract, commitment_contract)` is the only scheduling pipeline.
It contains no interval arithmetic and returns its actual intermediate contracts
for inspection. Both branches pass their availability contract to the same
synthesis helper. The old `schedule_reservations()` and independent `occupied`
calculation have been removed.

In [99]:
@dataclass(frozen=True)
class Analysis:
    projected_commitment: Contract | None = None
    projected_request: Contract | None = None
    strengthened_raw: Contract | None = None
    strengthened: Contract | None = None
    candidate: Contract | None = None
    candidate_consistent: bool | None = None
    weakened: Contract | None = None
    separated: Contract | None = None
    merged: Contract | None = None
    post_raw: Contract | None = None
    post: Contract | None = None
    availability: Contract | None = None
    synthesis: Synthesis = field(default_factory=lambda: Synthesis(P.empty(), ()))


def analyze(request_contract, commitment_contract):
    # This is the only scheduling orchestration. No interval algebra occurs here.
    view = request_view(request_contract)
    if view is None or view.starts.empty:
        return Analysis()
    kind, job_id, _ = _literal(view.literal)
    if kind != 'job':
        raise UnsupportedContractError('analysis starts in the original job vocabulary')
    if not consistent(commitment_contract):
        raise ValueError('existing commitments must be consistent')
    if any(_literal(lit)[1] == job_id for lit, _ in _commitment_coverage(commitment_contract)):
        raise ValueError('requested job must be new')

    with step('Project Request and Commitements'):
        cmt_p = project(commitment_contract, job_id)
        println(f'{cmt_p = !s}', '')
        req_p = project(request_contract, job_id)
        println(f'{req_p = !s}')

    with step('Strengthen and Decide Consistency'):
        strong_raw = strengthen_request(req_p)
        println(f'{strong_raw = !s}', '')
        strong = normalize(strong_raw)
        println(f'{strong = !s}', '')
        candidate = cmt_p.merge(strong)
        println(f'{candidate = !s}', '')
        passes = consistent(candidate)
        println(f'{passes = !s}')

    with step('Synthesize Offers'):
        weak = weaken_coverage(strong)
        println(f'{weak = !s}', '')
        separated = merged = raw = None
        if passes:
            with step(f'{passes=!s} => Direct Merge'):
                post = normalize(cmt_p.merge(weak))
                println(f'{post = !s}')
                available = weak
        else:
            with step(f'{passes=!s} => Repair'):
                separated = weak.separate(cmt_p)
                println(f'{separated = !s}', '')
                merged = separated.merge(cmt_p)
                println(f'{merged = !s}', '')
                raw = require_assumption(merged)
                println(f'{raw = !s}', '')
                post = normalize(raw)
                println(f'{post = !s}', '')
                available = extract_availability(post)
                println(f'{available = !s}')
        synthesis = synthesize(request_contract, available, commitment_contract)
        println(f'{synthesis.feasible_starts = !s}')

    return Analysis(
        projected_commitment=cmt_p, projected_request=req_p,
        strengthened_raw=strong_raw, strengthened=strong,
        candidate=candidate, candidate_consistent=passes, weakened=weak,
        separated=separated, merged=merged, post_raw=raw, post=post,
        availability=available, synthesis=synthesis,
    )

reset_step()
C_cmt = reservation(1, P.closedopen(F(2), F(3)))
C_req = Request(2, F(1), P.closedopen(F(1), F(5))).contract
analysis = analyze(C_req, C_cmt)


⏺ Step 1: Project Request and Commitements
│ cmt_p = ⟨ ⊤ ∣ □_[2,3) ¬p ⟩
│                           
│ req_p = ⟨ ⊤ ∣ ◇_[1,5) □_1 p ⟩
╯
⏺ Step 2: Strengthen and Decide Consistency
│ strong_raw = ⟨ ⊤ ∣ □_[1,5) □_1 p ⟩
│                                   
│ strong = ⟨ ⊤ ∣ □_[1,6) p ⟩
│                           
│ candidate = ⟨ ⊤ ∣ □_[1,6) p ∧ □_[2,3) ¬p ⟩
│                                           
│ passes = False
╯
⏺ Step 3: Synthesize Offers
│ weak = ⟨ ⊤ ∣ ◇_[1,6) p ⟩
│                         
├ ⏺ Step 3.1: passes=False => Repair
│ │ separated = ⟨ □_[1,6) ¬p ∨ □_[2,3) ¬p ∣ ◇_[1,6) p ⟩
│ │                                                    
│ │ merged = ⟨ □_[1,6) ¬p ∨ □_[2,3) ¬p ∣ ◇_[1,6) p ∧ ◇_[2,3) p ∨ ◇_[1,6) p ∧ □_[2,3) ¬p ⟩
│ │                                                                                      
│ │ raw = ⟨ ⊤ ∣ (◇_[1,6) p ∧ ◇_[2,3) p ∨ ◇_[1,6) p ∧ □_[2,3) ¬p) ∧ (□_[1,6) ¬p ∨ □_[2,3) ¬p) ⟩
│ │                                                                       

## 4. Inspect the synthesized offer contracts

Feasible starts and offers come from `synthesize(C_req, C_avail, C_cmt)`.
For this finite start set every surviving start is an offer. Each row below
queries a concrete contract; the display does not run a second interval scheduler.

In [100]:
alternatives = analysis.synthesis.offers
with step("Synthesize full-duration offer contracts"):
    println(f"Feasible starts: {fmt(analysis.synthesis.feasible_starts)}")
    for alternative in alternatives:
        println(str(alternative))
table(["Concrete allocation", "Refines request", "Consistent with commitments"], [
    (c, refines_request(c, C_req), consistent(C_cmt.merge(c))) for c in alternatives
])

⏺ Step 4: Synthesize full-duration offer contracts
│ Feasible starts: [1,1] ∪ [3,5)
│ ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩
│ ⟨ ⊤ ∣ □_[3,4) job = 2 ⟩
╯


| Concrete allocation | Refines request | Consistent with commitments |
| --- | --- | --- |
| ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩ | True | True |
| ⟨ ⊤ ∣ □_[3,4) job = 2 ⟩ | True | True |

## 5. Bargain over the fixed contract alternatives

Use $u^{EV}(C,q)=v_i(C)-q$, $u^{CH}(C,q)=q-c(C)$; $q$ is total price.
For the earlier/later offers, costs are 2/3 and EV values are 8/7. These
illustrative values correspond to 10 kWh, tariffs 0.20/0.30, and inconvenience
0/1 relative to an alternative costing 8. Floors are both 1, initial targets
are $(6,4)$ for (EV, charger), and the deadline is $T=2$.

The charger starts and sends all contract alternatives at its target price.
After rejection, both targets update by $t_{a,r+1}=t_{a,r}(1-r/T)$; this
illustration uses $\lambda_a=1$. Round-0 targets therefore stay unchanged.
Proposer floors filter offers without clamping targets. The evaluator chooses
highest utility, with earliest-start tie-breaking. Before the deadline it must
meet both target and floor; at the deadline it must **exceed** its floor.

`PricedOffer` retains the actual concrete contract. Costs, values, and price
metadata do not replace it with a start to be reconstructed later.

In [101]:
@dataclass(frozen=True)
class PricedOffer:
    contract: Contract
    price: F


def utility(party, offer, costs, values):
    return values[offer.contract] - offer.price if party == 'EV' else offer.price - costs[offer.contract]


def acceptable(u, target, floor, r, deadline):
    return u > floor if r == deadline else u >= target and u >= floor


def bargain_illustration(offers, costs, values, targets, minimum, deadline=2):
    """Fixed contracts and exact arithmetic; both concession exponents are 1."""
    if type(deadline) is not int or deadline <= 0:
        raise ValueError('deadline must be a positive integer')
    alternatives = tuple(sorted(offers, key=lambda c: offer_view(c).start))
    targets, history, proposer = dict(targets), [], 'CH'
    for r in range(deadline + 1):
        evaluator = 'EV' if proposer == 'CH' else 'CH'
        burst = tuple(PricedOffer(c, costs[c] + targets[proposer] if proposer == 'CH'
                                 else values[c] - targets[proposer]) for c in alternatives)
        burst = tuple(o for o in burst if utility(proposer, o, costs, values) >= minimum[proposer])
        row = dict(round=r, targets=dict(targets), proposer=proposer, evaluator=evaluator, offers=burst)
        if not burst:
            row.update(utilities=(), best=None, decision='No admissible offers')
            history.append(row)
            return None, history, 'no_admissible_offers'
        utilities = tuple(utility(evaluator, o, costs, values) for o in burst)
        index = max(range(len(burst)), key=lambda i: utilities[i])
        best = burst[index]
        accepted = acceptable(utilities[index], targets[evaluator], minimum[evaluator], r, deadline)
        row.update(utilities=utilities, best=best, decision='Accept' if accepted else 'Reject')
        history.append(row)
        if accepted:
            return best, history, 'accepted'
        if r == deadline:
            return None, history, 'deadline_rejection'
        targets = {a: target * (1 - F(r, deadline)) for a, target in targets.items()}
        proposer = evaluator

In [102]:
cost = dict(zip(alternatives, (F(2), F(3)), strict=True))
value = dict(zip(alternatives, (F(8), F(7)), strict=True))
floors = {'EV': F(1), 'CH': F(1)}
initial_targets = {'EV': F(6), 'CH': F(4)}
accepted, rounds, negotiation_status = bargain_illustration(
    alternatives, cost, value, initial_targets, floors
)
with step("Negotiate the fixed alternatives"):
    println(f"Result: {negotiation_status}", f"Accepted contract: {accepted.contract}")
    println(f"Total price: {accepted.price}")
table(["Round", "Targets (EV, CH)", "Proposer", "Offers (interval, price)",
       "Evaluator utilities", "Decision"], [
    (r['round'], (r['targets']['EV'], r['targets']['CH']), r['proposer'],
     '; '.join(f"({fmt(offer_view(o.contract).interval)}, {o.price})" for o in r['offers']),
     r['utilities'], r['decision']) for r in rounds
])
assert accepted == PricedOffer(alternatives[0], F(4))

⏺ Step 5: Negotiate the fixed alternatives
│ Result: accepted                          
│ Accepted contract: ⟨ ⊤ ∣ □_[1,2) job = 2 ⟩
│ Total price: 4
╯


| Round | Targets (EV, CH) | Proposer | Offers (interval, price) | Evaluator utilities | Decision |
| --- | --- | --- | --- | --- | --- |
| 0 | (6, 4) | CH | ([1,2), 6); ([3,4), 7) | (2, 0) | Reject |
| 1 | (6, 4) | EV | ([1,2), 2); ([3,4), 1) | (0, -2) | Reject |
| 2 | (3, 2) | CH | ([1,2), 4); ([3,4), 5) | (4, 2) | Accept |

## 6. Validate the accepted contract and merge it into commitments

Validate the accepted contract against the current request and commitments.
Require the purchased duration as well as refinement and consistency. On success,
update $C^{cmt\prime}=C^{cmt}\bullet C^{acc}$ and decode the stored record from
$C^{acc}$. Price is separate metadata. The original request fields are not used
to rebuild the accepted interval.

The helper below is sequential and in memory. Changed snapshots or failed
validation return `restart` and unchanged commitments. Unsupported contract
queries raise an explicit error. No agreement means no insertion.

**Missing production interface:** validation and insertion must be atomic over
the current request, commitments, and model inputs:

```text
snapshot := current request contract, commitment contract, and model inputs
offers := analyze(snapshot).synthesis.offers
if offers empty: end without agreement
agreement := negotiate(offers)
if no agreement: end without agreement
atomically:
    if current inputs differ from snapshot: restart scheduling
    if accepted contract fails refinement/duration/consistency: restart scheduling
    merge accepted contract into commitments
    store decoded job, interval, and price
```

The strengthened coverage, repaired availability, and unaccepted offers remain
temporary. Price rejection does not trigger rescheduling; changed inputs do.

In [103]:
@dataclass(frozen=True)
class StoredRecord:
    job: int
    interval: P.Interval
    price: F | None = None


def record_from_offer(contract, price=None):
    offer = offer_view(contract)
    return StoredRecord(offer.job, offer.interval, price)


def validate_commit_demo(current_cmt, current_req, snapshot_cmt, snapshot_req, agreement):
    """Sequential in-memory contract validation; not a transactional service."""
    if agreement is None:
        return 'no_agreement', current_cmt, None
    if current_cmt != snapshot_cmt or current_req != snapshot_req:
        return 'restart', current_cmt, None
    request, offer = request_view(current_req), offer_view(agreement.contract)
    if (request is None or offer.duration != request.duration
            or not refines_request(agreement.contract, current_req)
            or not consistent(current_cmt.merge(agreement.contract))):
        return 'restart', current_cmt, None
    if any(_literal(lit)[1] == offer.job for lit, _ in _commitment_coverage(current_cmt)):
        return 'restart', current_cmt, None
    updated = normalize(current_cmt.merge(agreement.contract))
    return 'committed', updated, record_from_offer(agreement.contract, agreement.price)

In [110]:
commit_status, C_committed, accepted_record = validate_commit_demo(
    C_cmt, C_req, C_cmt, C_req, accepted
)
stored_records = (record_from_offer(C_cmt), accepted_record)
with step("Revalidate and merge the accepted contract"):
    println(f"Status: {commit_status}", f"Updated commitment: {C_committed}")
    println(f"EV utility: {utility('EV', accepted, cost, value)}")
    println(f"Charger utility: {utility('CH', accepted, cost, value)}")
    println(f"Projection for a later job 3: {normalize(project(C_committed, 3))}")
table(["Stored job", "Interval decoded from accepted contract", "Price metadata"], [
    (r.job, r.interval, 'Not supplied' if r.price is None else r.price) for r in stored_records
])
assert commit_status == 'committed'
assert C_committed == normalize(C_cmt.merge(accepted.contract))
assert accepted_record == StoredRecord(2, P.closedopen(1, 2), F(4))
assert utility('EV', accepted, cost, value) == 4
assert utility('CH', accepted, cost, value) == 2

⏺ Step 12: Revalidate and merge the accepted contract
│ Status: committed                                            
│ Updated commitment: ⟨ ⊤ ∣ □_[1,2) job = 2 ∧ □_[2,3) job = 1 ⟩
│ EV utility: 4
│ Charger utility: 2
│ Projection for a later job 3: ⟨ ⊤ ∣ □_[1,3) ¬p ⟩
╯


| Stored job | Interval decoded from accepted contract | Price metadata |
| --- | --- | --- |
| 1 | [2,3) | Not supplied |
| 2 | [1,2) | 4 |